# Upload Complete Model Directory to HuggingFace

This notebook uploads a ready-to-use Chatterbox model directory such as `final_model`. It validates the runtime files needed by the selected variant and excludes Hugging Face Trainer export artifacts before upload.

## 1. Configuration

Set the local model directory and target Hugging Face repository.

In [ ]:
MODEL_VARIANT = "multilingual"  # Options: "base", "multilingual", "turbo"
model_path = "final_model"  # UPDATE THIS
hf_repo_name = "upload/path"  # UPDATE THIS
private_repo = True

EXTRA_EXCLUDE_FILES = []


## 2. Import Libraries

In [ ]:
import shutil
import tempfile
from pathlib import Path

from huggingface_hub import HfApi, create_repo, upload_folder

REQUIRED_FILES = {
    "base": [
        "ve.safetensors",
        "t3_cfg.safetensors",
        "s3gen.safetensors",
        "tokenizer.json",
    ],
    "multilingual": [
        "ve.pt",
        "t3_mtl23ls_v2.safetensors",
        "s3gen.pt",
        "grapheme_mtl_merged_expanded_v1.json",
        "Cangjie5_TC.json",
    ],
    "turbo": [
        "ve.safetensors",
        "t3_turbo_v1.safetensors",
        "s3gen_meanflow.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
    ],
}

OPTIONAL_FILES = {
    "base": ["conds.pt"],
    "multilingual": ["conds.pt"],
    "turbo": ["conds.pt", "merges.txt", "vocab.json"],
}

HF_EXPORT_ARTIFACTS = {
    "model.safetensors",
    "training_args.bin",
    "config.json",
    "generation_config.json",
}


## 3. Validate the Model Directory

In [ ]:
model_dir = Path(model_path).expanduser().resolve()
assert MODEL_VARIANT in REQUIRED_FILES, f"Unsupported MODEL_VARIANT: {MODEL_VARIANT}"
assert model_dir.exists(), f"Model directory not found: {model_dir}"
assert model_dir.is_dir(), f"Expected a directory, got: {model_dir}"

required_files = REQUIRED_FILES[MODEL_VARIANT]
optional_files = OPTIONAL_FILES[MODEL_VARIANT]
missing_files = [name for name in required_files if not (model_dir / name).exists()]
assert not missing_files, f"Missing required files for {MODEL_VARIANT}: {missing_files}"

present_optional_files = [name for name in optional_files if (model_dir / name).exists()]
present_hf_artifacts = [name for name in sorted(HF_EXPORT_ARTIFACTS) if (model_dir / name).exists()]

print(f"✓ Model directory: {model_dir}")
print(f"✓ Variant: {MODEL_VARIANT}")
print(f"✓ Required files present: {required_files}")
print(f"✓ Optional files present: {present_optional_files}")
if present_hf_artifacts:
    print(f"Will exclude Hugging Face export artifacts: {present_hf_artifacts}")
else:
    print("✓ No Hugging Face export artifacts detected")


## 4. Stage a Clean Upload Directory

Copy the model directory to a temporary location and remove files that should not be uploaded.

In [ ]:
temp_dir = Path(tempfile.mkdtemp(prefix="hf_upload_model_"))
print(f"Created temporary directory: {temp_dir}")

upload_dir = temp_dir / model_dir.name
shutil.copytree(model_dir, upload_dir, dirs_exist_ok=True)
print(f"✓ Copied model directory to {upload_dir}")

excluded_files = sorted(set(HF_EXPORT_ARTIFACTS).union(EXTRA_EXCLUDE_FILES))
removed_files = []
for file_name in excluded_files:
    file_path = upload_dir / file_name
    if file_path.exists():
        file_path.unlink()
        removed_files.append(file_name)

print(f"✓ Removed excluded files: {removed_files}")
print(f"Files ready for upload: {[path.name for path in sorted(upload_dir.iterdir()) if path.is_file()]}")


## 5. Create the Hugging Face Repository

In [ ]:
api = HfApi()

create_repo(
    repo_id=hf_repo_name,
    private=private_repo,
    exist_ok=True,
)
print(f"✓ Repository '{hf_repo_name}' ready (private={private_repo})")


## 6. Upload the Model Directory

In [ ]:
print(f"Uploading model directory to {hf_repo_name}...")

upload_folder(
    folder_path=upload_dir,
    repo_id=hf_repo_name,
    repo_type="model",
    commit_message="Upload complete Chatterbox model directory",
)

print(f"\n✓ Successfully uploaded to https://huggingface.co/{hf_repo_name}")


## 7. Cleanup

In [ ]:
print(f"Cleaning up temporary directory: {temp_dir}")
shutil.rmtree(temp_dir)
print("✓ Cleanup complete")
